In [ ]:
import os
import glob
import pandas as pd

# ==========================================
# SETTINGS
# ==========================================
FOLDER_PATH = "Data/match/"                     # The folder where your scraped player files are
OUTPUT_DIR = "Data/Sessions/SessionGames"       # The folder where the new files will go
SESSION_THRESHOLD_MS = 2 * 60 * 60 * 1000       # 2 hours in milliseconds
MIN_SESSION_LENGTH = 1                          # Do not save sessions with fewer games than this

print(f"Scanning folder: {FOLDER_PATH} for CSV files...")

# Find all CSV files in the folder
file_pattern = os.path.join(FOLDER_PATH, "*.csv")
match_files = glob.glob(file_pattern)

print(f"Found {len(match_files)} player files. Processing...")

# Create the output folder if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

session_count = 0
processed_count = 0

for file in match_files:
    try:
        df = pd.read_csv(file)
        
        if df.empty or 'game_start_timestamp' not in df.columns or 'win' not in df.columns or 'puuid' not in df.columns:
            continue
            
        df = df.sort_values(by='game_start_timestamp').reset_index(drop=True)
        
        player_id = df['puuid'].iloc[0]
        df['time_diff'] = df['game_start_timestamp'].diff()
        
        # Skapar sessioner baserat på tidsgapet mellan spel. 
        df['is_new_session'] = (df['time_diff'] > SESSION_THRESHOLD_MS) | df['time_diff'].isna()
        df['session_id'] = df['is_new_session'].cumsum()
        
        grouped_sessions = df.groupby('session_id')

        saved_session_dfs = []
        saved_session_count = 0

        for session_id, session_df in grouped_sessions:
            # Kronologisk sortering av spel inom sessionen 
            session_df = session_df.sort_values(by='game_start_timestamp').reset_index(drop=True)

            session_df['game_in_session'] = session_df.index + 1

            session_length = len(session_df)

            if session_length < MIN_SESSION_LENGTH:
                continue

            saved_session_count += 1
            session_df['session_number_for_player'] = saved_session_count

            session_df = session_df.drop(columns=['time_diff', 'is_new_session', 'session_id'])

            saved_session_dfs.append(session_df)

        if saved_session_dfs:
            player_sessions_df = pd.concat(saved_session_dfs, ignore_index=True)
            filename = f"player_{player_id}_sessions_{saved_session_count}.csv"
            filepath = os.path.join(OUTPUT_DIR, filename)
            player_sessions_df.to_csv(filepath, index=False)
            session_count += saved_session_count
        
        processed_count += 1
        
    except Exception as e:
        print(f"Error processing {file}: {e}")

print(f"\nSUCCESS! Processed {processed_count} players.")
print(f"Saved {session_count} total sessions to: {OUTPUT_DIR}")


Scanning folder: Data/match/ for CSV files...
Found 1386 player files. Processing...

SUCCESS! Processed 1386 players.
Saved 124111 total sessions to: Data/Sessions/SessionGames
